# Somo 08 - Mfano wa Muundo wa Wakala Wengi


## Mipangilio


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Kwanini Mifumo ya Wakala Wengi?

Kazi halisi kama kupanga safari zinahusisha aina nyingi tofauti za utaalamu — usafirishaji, maarifa ya eneo, bajeti, na zaidi. Wakala mmoja anayejaribu kushughulikia kila kitu haraka huwa mgumu kudhibiti.

Mifumo ya wakala wengi hutatua hili kupitia **utaalamu maalum**: kila wakala anazingatia eneo moja la utaalamu, akitoa matokeo bora kuliko mlezi wa jumla. Pia huboresha **uwezo wa kupanuka** — unaweza kuongeza wakala wapya (mfano, mtaalamu wa ndege, mkosoaji wa migahawa) bila kuandika upya mfumo uliopo. Wakala huungana kupitia njia iliyo pangiliwa, wanapita muktadha kutoka kwa mmoja hadi mwingine.


## Kuunda Wakala Maalum


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Kujenga Mtiririko wa Kazi wa Mfululizo

`WorkflowBuilder` hukuruhusu kuunganisha maajenti katika mchoro ulioratibiwa. Hapa tunaunda mchakato rahisi wa hatua mbili: **TravelPlanner** huandaa ratiba, kisha **TravelConcierge** hukagua na kuiboresha.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Kuongeza Maajenti Zaidi Kwenye Mtiririko wa Kazi

Moja ya faida kuu ya muundo wa maajenti wengi ni jinsi ulivyo rahisi kuongezwa. Hapa chini tunaongeza wakala wa **BudgetReviewer** anayechunguza mpango dhidi ya bajeti ya msafiri, kuonyesha vitu ambavyo vinaweza kusababisha gharama kuzidi kikomo, na kupendekeza mbadala za kuokoa fedha. Mtiririko wa kazi sasa unaendesha maajenti watatu mfululizo:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Muhtasari

Katika somo hili ulijifunza jinsi ya:

1. **Kuunda mawakala maalum** — kila mmoja akiwa na jukumu lililoelekezwa (upangaji, huduma kwa wateja, ukaguzi wa bajeti).
2. **Kuwaunganisha mawakala katika mtiririko wa kazi wa mfululizo** kwa kutumia `WorkflowBuilder` na `add_edge`.
3. **Kusambaza matokeo** kutoka kwenye mchakato wa mawakala wengi, ukifuata wakala anayeongea.
4. **Kupanua mtiririko wa kazi** kwa kuongeza mawakala wapya kwenye mnyororo bila kubadilisha mawakala waliopo.

Mfumo wa wabunifu wa mawakala wengi unahakikisha kila wakala ni rahisi huku ukitengeneza matokeo tajiri zaidi, yaliyoangaliwa kwa undani kuliko yanavyoweza kufikiwa na wakala mmoja peke yake.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Hatia ya Msalaba**:  
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kuwa tafsiri za kiotomatiki zinaweza kuwa na makosa au upotovu wa maana. Nyaraka asili katika lugha yake ya asili inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, inashauriwa kutumia tafsiri ya kitaalamu inayofanywa na binadamu. Hatuwajibiki kwa kutoelewana au tafsiri potofu zitokanazo na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
